# InsightFlow — Database Connection

This notebook establishes a connection between the InsightFlow PostgreSQL database and Python using SQLAlchemy and Pandas.

## 1. Import Libraries

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

## 2. Database Configuration

In [3]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

In [4]:
print("User:", DB_USER)
print("Host:", DB_HOST)
print("Port:", DB_PORT)
print("Database:", DB_NAME)
print("Password loaded:", DB_PASSWORD is not None)

User: postgres
Host: localhost
Port: 5432
Database: insightflow_db
Password loaded: True


## 3. PostgreSQL Connection

In [5]:
DATABASE_URL = (
    f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

In [6]:
with engine.connect() as connection:
    print("Database connection successful.")

Database connection successful.


## 4. Load PostgreSQL Tables into Pandas

In [7]:
customers = pd.read_sql(
    "SELECT * FROM customers;",
    engine
)

products = pd.read_sql(
    "SELECT * FROM products;",
    engine
)

orders = pd.read_sql(
    "SELECT * FROM orders;",
    engine
)

order_items = pd.read_sql(
    "SELECT * FROM order_items;",
    engine
)

payments = pd.read_sql(
    "SELECT * FROM payments;",
    engine
)

categories = pd.read_sql(
    "SELECT * FROM categories;",
    engine
)

In [8]:
customers.head()

,customer_id,first_name,last_name,email,city,country,signup_date
0,1,Emma,Johnson,emma.johnson@example.com,London,United Kingdom,2025-01-15
1,2,Liam,Smith,liam.smith@example.com,Manchester,United Kingdom,2025-02-03
2,3,Sofia,Rossi,sofia.rossi@example.com,Milan,Italy,2025-02-21
3,4,Lucas,Martin,lucas.martin@example.com,Paris,France,2025-03-10
4,5,Anna,Schmidt,anna.schmidt@example.com,Berlin,Germany,2025-03-27


In [9]:
customers.shape

(11, 7)

In [10]:
customers.columns

Index(['customer_id', 'first_name', 'last_name', 'email', 'city', 'country',
       'signup_date'],
      dtype='str')

In [11]:
customers.dtypes

customer_id     int64
first_name        str
last_name         str
email             str
city              str
country           str
signup_date    object
dtype: object

In [12]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  11 non-null     int64 
 1   first_name   11 non-null     str   
 2   last_name    11 non-null     str   
 3   email        11 non-null     str   
 4   city         10 non-null     str   
 5   country      11 non-null     str   
 6   signup_date  11 non-null     object
dtypes: int64(1), object(1), str(5)
memory usage: 748.0+ bytes


In [13]:
customers.isna().sum()

customer_id    0
first_name     0
last_name      0
email          0
city           1
country        0
signup_date    0
dtype: int64

In [14]:
products.shape

(10, 6)

In [15]:
products.describe()

,product_id,category_id,cost,list_price,stock_quantity
count,10.00000,10.000000,10.000000,10.000000,10.000000
mean,5.50000,2.600000,40.500000,83.490000,95.500000
std,3.02765,1.429841,25.777898,49.668344,49.297848
min,1.00000,1.000000,8.000000,24.990000,40.000000
25%,3.25000,1.250000,16.250000,32.490000,61.250000
50%,5.50000,2.500000,42.500000,89.990000,82.500000
75%,7.75000,3.750000,58.750000,117.490000,115.000000
max,10.00000,5.000000,80.000000,159.990000,200.000000


## 5. Create an Analytical Dataset

This section combines transactional data from multiple PostgreSQL tables into a single DataFrame for analysis.

In [19]:
query = """
SELECT
    o.order_id,
    o.order_date,
    o.status,
    c.customer_id,
    c.first_name,
    c.last_name,
    c.country,
    p.product_id,
    p.product_name,
    cat.category_name,
    oi.quantity,
    oi.unit_price,
    oi.discount
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
JOIN order_items AS oi
    ON o.order_id = oi.order_id
JOIN products AS p
    ON oi.product_id = p.product_id
JOIN categories AS cat
    ON p.category_id = cat.category_id;
"""

sales = pd.read_sql(query, engine)

In [20]:
sales.head()

,order_id,order_date,status,customer_id,first_name,last_name,country,product_id,product_name,category_name,quantity,unit_price,discount
0,1,2025-06-02,Completed,1,Emma,Johnson,United Kingdom,1,Wireless Headphones,Electronics,1,89.99,0.00
1,1,2025-06-02,Completed,1,Emma,Johnson,United Kingdom,6,Data Science Handbook,Books,1,39.99,0.10
2,2,2025-06-03,Completed,2,Liam,Smith,United Kingdom,2,Mechanical Keyboard,Electronics,1,109.99,0.00
3,3,2025-06-04,Completed,3,Sofia,Rossi,Italy,4,Coffee Machine,Home & Kitchen,1,139.99,0.15
4,3,2025-06-04,Completed,3,Sofia,Rossi,Italy,8,Yoga Mat,Sports,2,29.99,0.00


### Revenue Calculation

In [21]:
sales["revenue"] = (
    sales["quantity"]
    * sales["unit_price"]
    * (1 - sales["discount"])
)

In [22]:
sales[
    ["product_name", "quantity", "unit_price", "discount", "revenue"]
].head()

,product_name,quantity,unit_price,discount,revenue
0,Wireless Headphones,1,89.99,0.00,89.9900
1,Data Science Handbook,1,39.99,0.10,35.9910
2,Mechanical Keyboard,1,109.99,0.00,109.9900
3,Coffee Machine,1,139.99,0.15,118.9915
4,Yoga Mat,2,29.99,0.00,59.9800


In [23]:
completed_sales = sales[
    sales["status"] == "Completed"
].copy()

In [24]:
completed_sales.head()

,order_id,order_date,status,customer_id,first_name,last_name,country,product_id,product_name,category_name,quantity,unit_price,discount,revenue
0,1,2025-06-02,Completed,1,Emma,Johnson,United Kingdom,1,Wireless Headphones,Electronics,1,89.99,0.00,89.9900
1,1,2025-06-02,Completed,1,Emma,Johnson,United Kingdom,6,Data Science Handbook,Books,1,39.99,0.10,35.9910
2,2,2025-06-03,Completed,2,Liam,Smith,United Kingdom,2,Mechanical Keyboard,Electronics,1,109.99,0.00,109.9900
3,3,2025-06-04,Completed,3,Sofia,Rossi,Italy,4,Coffee Machine,Home & Kitchen,1,139.99,0.15,118.9915
4,3,2025-06-04,Completed,3,Sofia,Rossi,Italy,8,Yoga Mat,Sports,2,29.99,0.00,59.9800


In [25]:
completed_sales["revenue"].sum()

np.float64(1429.0275000000001)

In [26]:
product_revenue = (
    completed_sales
    .groupby("product_name")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

product_revenue

product_name
Coffee Machine           258.9815
Air Fryer                233.9805
Mechanical Keyboard      219.9800
Smart Watch              127.9920
Basic T-Shirt            121.2015
Data Science Handbook    115.9710
Wireless Headphones       89.9900
SQL Fundamentals          89.9700
Yoga Mat                  89.9700
Running Shoes             80.9910
Name: revenue, dtype: float64

In [29]:
completed_sales = sales[
    sales["status"] == "Completed"
].copy()

In [30]:
completed_sales.shape

(17, 14)

In [31]:
category_revenue = (
    completed_sales
    .groupby("category_name")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

category_revenue

category_name
Home & Kitchen    492.9620
Electronics       437.9620
Books             205.9410
Sports            170.9610
Clothing          121.2015
Name: revenue, dtype: float64

In [32]:
product_quantity = (
    completed_sales
    .groupby("product_name")["quantity"]
    .sum()
    .sort_values(ascending=False)
)

product_quantity

product_name
Basic T-Shirt            5
Data Science Handbook    3
Yoga Mat                 3
SQL Fundamentals         3
Air Fryer                2
Coffee Machine           2
Mechanical Keyboard      2
Running Shoes            1
Smart Watch              1
Wireless Headphones      1
Name: quantity, dtype: int64

## 6. Aggregation with Pandas

This section summarizes product performance using multiple aggregation metrics.

In [33]:
product_summary = (
    completed_sales
    .groupby("product_name")
    .agg(
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        average_unit_price=("unit_price", "mean")
    )
)

product_summary

,total_revenue,units_sold,average_unit_price
product_name,,,
Air Fryer,233.9805,2,119.99
Basic T-Shirt,121.2015,5,24.99
Coffee Machine,258.9815,2,139.99
Data Science Handbook,115.9710,3,39.99
Mechanical Keyboard,219.9800,2,109.99
Running Shoes,80.9910,1,89.99
SQL Fundamentals,89.9700,3,29.99
Smart Watch,127.9920,1,159.99
Wireless Headphones,89.9900,1,89.99


In [34]:
product_summary = (
    completed_sales
    .groupby("product_name")
    .agg(
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        average_unit_price=("unit_price", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

product_summary

,total_revenue,units_sold,average_unit_price
product_name,,,
Coffee Machine,258.9815,2,139.99
Air Fryer,233.9805,2,119.99
Mechanical Keyboard,219.9800,2,109.99
Smart Watch,127.9920,1,159.99
Basic T-Shirt,121.2015,5,24.99
Data Science Handbook,115.9710,3,39.99
Wireless Headphones,89.9900,1,89.99
SQL Fundamentals,89.9700,3,29.99
Yoga Mat,89.9700,3,29.99


In [35]:
category_summary = (
    completed_sales
    .groupby("category_name")
    .agg(
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        average_unit_price=("unit_price", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

category_summary

,total_revenue,units_sold,average_unit_price
category_name,,,
Home & Kitchen,492.9620,4,129.99
Electronics,437.9620,4,117.49
Books,205.9410,6,34.99
Sports,170.9610,4,49.99
Clothing,121.2015,5,24.99


## 7. Initial Exploratory Data Analysis

This section performs a preliminary exploration of the transactional data before the full data cleaning and EDA process.

In [36]:
print("Number of sales rows:", len(completed_sales))
print("Number of customers:", completed_sales["customer_id"].nunique())
print("Number of products:", completed_sales["product_id"].nunique())
print("Number of categories:", completed_sales["category_name"].nunique())
print("Number of orders:", completed_sales["order_id"].nunique())

Number of sales rows: 17
Number of customers: 8
Number of products: 10
Number of categories: 5
Number of orders: 10


In [37]:
total_revenue = completed_sales["revenue"].sum()
total_orders = completed_sales["order_id"].nunique()
total_units = completed_sales["quantity"].sum()

average_order_value = total_revenue / total_orders

print(f"Total Revenue: {total_revenue:.2f}")
print(f"Total Orders: {total_orders}")
print(f"Total Units Sold: {total_units}")
print(f"Average Order Value: {average_order_value:.2f}")

Total Revenue: 1429.03
Total Orders: 10
Total Units Sold: 23
Average Order Value: 142.90


In [38]:
kpi_summary = pd.Series({
    "Total Revenue": total_revenue,
    "Total Orders": total_orders,
    "Total Units Sold": total_units,
    "Average Order Value": average_order_value
})

kpi_summary

Total Revenue          1429.02750
Total Orders             10.00000
Total Units Sold         23.00000
Average Order Value     142.90275
dtype: float64

### Initial Data Quality Check

In [39]:
completed_sales.isna().sum()

order_id         0
order_date       0
status           0
customer_id      0
first_name       0
last_name        0
country          0
product_id       0
product_name     0
category_name    0
quantity         0
unit_price       0
discount         0
revenue          0
dtype: int64

In [40]:
completed_sales.duplicated().sum()

np.int64(0)

In [41]:
completed_sales.dtypes

order_id           int64
order_date        object
status               str
customer_id        int64
first_name           str
last_name            str
country              str
product_id         int64
product_name         str
category_name        str
quantity           int64
unit_price       float64
discount         float64
revenue          float64
dtype: object

## 8. Initial Findings

- The PostgreSQL database was successfully connected to Python using SQLAlchemy.
- PostgreSQL tables were loaded into Pandas DataFrames.
- Transactional tables were joined to create a unified sales dataset.
- Revenue was calculated at the transaction-line level.
- Completed orders were separated for revenue analysis.
- Product, category, and country-level aggregations were created using Pandas.
- Initial KPI and data-quality checks were performed.

> Note: This notebook uses the small synthetic development dataset. The full data cleaning and exploratory analysis will be performed on a real e-commerce dataset in the next stage.